⚽ Proxecto Power BI: Análise de Datos de Fútbol

🎯 Obxectivo

Analizar estatísticas de equipos e xogadores de fútbol, cruzando datos de distintas fontes para obter información valiosa sobre rendemento, condicións meteorolóxicas durante os partidos e outros factores relevantes.

📁 Fontes de Datos

    1. Excel: Estatísticas de equipos (partidos xogados, vitorias, empates, derrotas, goles a favor e en contra).

    2. JSON: Datos meteorolóxicos durante os partidos, obtidos mediante scraping.

    3. Script de Python: Para realizar o scraping dos datos meteorolóxicos e procesalos.

    4. Spark-HDFS: Datos históricos de partidos, almacenados en formato Parquet.

🐍 Script de Scraping en Python

Utilizaremos requests e BeautifulSoup para obter datos meteorolóxicos de partidos desde unha fonte como Time and Date.

In [1]:
import requests
from bs4 import BeautifulSoup
import json
import pandas as pd

# URL da páxina de meteoroloxía
url = "https://www.timeanddate.com/weather/spain"

response = requests.get(url)
soup = BeautifulSoup(response.content, 'html.parser')

cities_data = []

for row in soup.select("table tbody tr"):
    cols = row.find_all("td")
    if len(cols) >= 2:
        city = cols[0].text.strip()
        temp = cols[1].text.strip().replace("°C", "")
        try:
            temp = float(temp)
        except:
            temp = None
        cities_data.append({"City": city, "Temperature": temp})

# Gardar en JSON
with open("weather_spain.json", "w", encoding='utf-8') as f:
    json.dump(cities_data, f, ensure_ascii=False, indent=2)

# Tamén gardar en CSV (opcional)
pd.DataFrame(cities_data).to_csv("weather_spain.csv", index=False)


PermissionError: [Errno 13] Permission denied: 'weather_spain.json'

Este script recolle as temperaturas das principais cidades de España e gárdaas en formato JSON e CSV para a súa posterior análise.

🐼 Procesamento de Datos con Pandas en Power BI

In [ ]:
import pandas as pd
import json

# Ler JSON exportado do scraping
with open("weather_spain.json", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)

# Limpeza básica
df = df.dropna()
df["City"] = df["City"].str.strip().str.title()

df

Este script limpa os datos meteorolóxicos e prepáraos para a súa integración con outras fontes de datos en Power BI.

🔗 Integración con Spark-HDFS

Para integrar datos almacenados en HDFS, podemos utilizar PySpark:

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("FootballData").getOrCreate()

# Ler datos de partidos desde HDFS
df = spark.read.parquet("hdfs://localhost:9000/datos_futbol/partidos.parquet")
df.show()


Estes datos poden incluír información detallada sobre partidos, como resultados, estatísticas de xogadores, etc.